In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("food_sales.csv")

In [3]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6504 entries, 0 to 6503
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   日期      6504 non-null   object 
 1   菜品      6504 non-null   object 
 2   销量      6504 non-null   float64
dtypes: float64(1), object(2)
memory usage: 152.6+ KB


In [4]:
df["日期"] = pd.to_datetime(df["日期"])

In [5]:
# Same date + same dish: sum sales into one row
df = df.groupby(["日期", "菜品"], as_index=False)["销量"].sum()
df = df.sort_values(["日期", "菜品"], na_position="last")
df.head()

,日期,菜品,销量
0,2025-05-01,掌中宝（2串起烤）,51.0
1,2025-05-01,掌中宝（外卖）,15.0
2,2025-05-01,海鲈鱼,3.0
3,2025-05-01,烤羊肉串（2串起烤）,168.0
4,2025-05-01,翡翠青椒,25.0


In [6]:
# Normalize inconsistent names for 'Zhangzhongbao'
mask = df["菜品"].str.contains("掌中宝", na=False)
df.loc[mask, "菜品"] = "掌中宝"
df = df.groupby(["日期", "菜品"], as_index=False)["销量"].sum()

In [7]:
df.head()

,日期,菜品,销量
0,2025-05-01,掌中宝,66.0
1,2025-05-01,海鲈鱼,3.0
2,2025-05-01,烤羊肉串（2串起烤）,168.0
3,2025-05-01,翡翠青椒,25.0
4,2025-05-01,老坛酸菜,8.0


In [8]:
# Normalize the name for 'grilled lamb skewers'
df.loc[df["菜品"] == "烤羊肉串（2串起烤）", "菜品"] = "烤羊肉串"

In [9]:
df.head()

,日期,菜品,销量
0,2025-05-01,掌中宝,66.0
1,2025-05-01,海鲈鱼,3.0
2,2025-05-01,烤羊肉串,168.0
3,2025-05-01,翡翠青椒,25.0
4,2025-05-01,老坛酸菜,8.0


In [10]:
df["菜品"].unique()

array(['掌中宝', '海鲈鱼', '烤羊肉串', '翡翠青椒', '老坛酸菜', '蒜香味', '蒜香豆花黔鱼（外卖）', '豆花海鲈鱼',
       '豆花黔鱼', '酸菜豆花烤鱼（外卖）', '青柠酸辣', '青椒味小黔鱼（外卖）', '青椒豆花烤鱼（外卖）',
       '麻辣烤鱼最佳搭配（卤味豆干+青笋条+土豆片）', '麻辣烤鱼肥肠组合（青笋+卤肥肠+土豆片）', '麻辣豆花烤鱼（外卖）',
       '烤羊肉串（外卖）', '江团', '麻辣豆花黔鱼（外卖）', '青椒豆花黔鱼（外卖）', '黔鱼', '小份麻辣味烤鱼（黑豆花）',
       '麻辣小黔鱼（外卖）', '小份蒜香味烤鱼（黑豆花）', '小份青椒味烤鱼（黑豆花）', '豆花江团',
       '小份酸菜味烤鱼（黑豆花）', '酸菜味小黔鱼（外卖）', '麻辣烤鱼肥肠组合（青笋条+卤肥肠+土豆片）',
       '麻辣烤鱼灵魂搭配：卤味豆干+青笋片+土豆片', '酸菜豆花黔鱼（外卖）', '经典麻辣豆花江团', '小份红灯笼味烤鱼（黑豆花）',
       '大红灯笼味豆花黔鱼（外卖）', '大红灯笼泡椒味', '经典麻辣味', '蒜香味豆花海鲈鱼（外卖）',
       '酸菜味豆花海鲈鱼（外卖）', '青椒味豆花海鲈鱼（外卖）', '麻辣味豆花海鲈鱼（外卖）'], dtype=object)

In [11]:
# Remove 'douhua' from dish names (e.g., 'douhua sea bass' -> 'sea bass')
mask2 = df["菜品"].str.contains("豆花", na=False)
df.loc[mask2, "菜品"] = df.loc[mask2, "菜品"].str.replace("豆花", "", regex=False).str.strip()
# If a dish name is exactly 'douhua', restore it after removal to avoid empty strings
df.loc[df["菜品"] == "", "菜品"] = "豆花"

In [12]:
# Remove '(takeout)' from dish names (e.g., 'green pepper grilled fish (takeout)' -> 'green pepper grilled fish')
mask3 = df["菜品"].str.contains("（外卖）", na=False)
df.loc[mask3, "菜品"] = df.loc[mask3, "菜品"].str.replace("（外卖）", "", regex=False).str.strip()

In [13]:
# Remove extra suffixes after 'mala grilled fish' (e.g., '(bamboo shoots + braised intestines + potato slices)')
mask4 = df["菜品"].str.contains("麻辣烤鱼", na=False)
df.loc[mask4, "菜品"] = "麻辣烤鱼"

In [14]:
# Remove '(black)' from dish names (e.g., 'small green pepper grilled fish (black)' -> 'small green pepper grilled fish')
mask5 = df["菜品"].str.contains("（黑）", na=False)
df.loc[mask5, "菜品"] = df.loc[mask5, "菜品"].str.replace("（黑）", "", regex=False).str.strip()

In [15]:
# Remove 'small' from dish names (e.g., 'small garlic grilled fish' -> 'garlic grilled fish')
mask6 = df["菜品"].str.contains("小份", na=False)
df.loc[mask6, "菜品"] = df.loc[mask6, "菜品"].str.replace("小份", "", regex=False).str.strip()

In [16]:
# Remove 'small' from dish names (e.g., 'garlic small Qian fish' -> 'garlic Qian fish')
mask7 = df["菜品"].str.contains("小", na=False)
df.loc[mask7, "菜品"] = df.loc[mask7, "菜品"].str.replace("小", "", regex=False).str.strip()

In [17]:
# Split unflavored 'sea bass' by date into 4 flavors: garlic/pickled mustard greens/green pepper/mala
flavors = ["蒜香味海鲈鱼", "酸菜味海鲈鱼", "青椒味海鲈鱼", "麻辣味海鲈鱼"]

base = (
    df.loc[df["菜品"] == "海鲈鱼", ["日期", "销量"]].groupby("日期", as_index=False)["销量"].sum()
)
base["销量"] = base["销量"] / 4

new_rows = pd.concat([base.assign(菜品=flv) for flv in flavors], ignore_index=True)

df = df[df["菜品"] != "海鲈鱼"]
df = pd.concat([df, new_rows], ignore_index=True)

In [18]:
# Split unflavored 'Qian fish' by date into 4 flavors: garlic/pickled mustard greens/green pepper/mala
flavors = ["蒜香味黔鱼", "酸菜味黔鱼", "青椒味黔鱼", "麻辣味黔鱼"]

base = df.loc[df["菜品"] == "黔鱼", ["日期", "销量"]].groupby("日期", as_index=False)["销量"].sum()
base["销量"] = base["销量"] / 4

new_rows = pd.concat([base.assign(菜品=flv) for flv in flavors], ignore_index=True)

df = df[df["菜品"] != "黔鱼"]
df = pd.concat([df, new_rows], ignore_index=True)

In [19]:
# Split unflavored 'Jiangtuan' by date into 4 flavors: garlic/pickled mustard greens/green pepper/mala
flavors = ["蒜香味江团", "酸菜味江团", "青椒味江团", "麻辣味江团"]

base = df.loc[df["菜品"] == "江团", ["日期", "销量"]].groupby("日期", as_index=False)["销量"].sum()
base["销量"] = base["销量"] / 4

new_rows = pd.concat([base.assign(菜品=flv) for flv in flavors], ignore_index=True)

df = df[df["菜品"] != "江团"]
df = pd.concat([df, new_rows], ignore_index=True)

In [20]:
# Split 'emerald green pepper' (no fish type) by date into 3 fish types: Qian fish/Jiangtuan/sea bass
flavors = ["青椒味黔鱼", "青椒味江团", "青椒味海鲈鱼"]

base = (
    df.loc[df["菜品"] == "翡翠青椒", ["日期", "销量"]].groupby("日期", as_index=False)["销量"].sum()
)
base["销量"] = base["销量"] / 3

new_rows = pd.concat([base.assign(菜品=flv) for flv in flavors], ignore_index=True)

df = df[df["菜品"] != "翡翠青椒"]
df = pd.concat([df, new_rows], ignore_index=True)

In [21]:
# Split 'old-jar pickled mustard greens' (no fish type) by date into 3 fish types: Qian fish/Jiangtuan/sea bass
flavors = ["老坛酸菜味黔鱼", "老坛酸菜味江团", "老坛酸菜味海鲈鱼"]

base = (
    df.loc[df["菜品"] == "老坛酸菜", ["日期", "销量"]].groupby("日期", as_index=False)["销量"].sum()
)
base["销量"] = base["销量"] / 3

new_rows = pd.concat([base.assign(菜品=flv) for flv in flavors], ignore_index=True)

df = df[df["菜品"] != "老坛酸菜"]
df = pd.concat([df, new_rows], ignore_index=True)

In [22]:
# Split 'garlic flavor' (no fish type) by date into 3 fish types: Qian fish/Jiangtuan/sea bass
flavors = ["蒜香味黔鱼", "蒜香味江团", "蒜香味海鲈鱼"]

base = (
    df.loc[df["菜品"] == "蒜香味", ["日期", "销量"]].groupby("日期", as_index=False)["销量"].sum()
)
base["销量"] = base["销量"] / 3

new_rows = pd.concat([base.assign(菜品=flv) for flv in flavors], ignore_index=True)

df = df[df["菜品"] != "蒜香味"]
df = pd.concat([df, new_rows], ignore_index=True)

In [23]:
# Split 'mala grilled fish' (no fish type) by date into 3 fish types: Qian fish/Jiangtuan/sea bass
flavors = ["麻辣味黔鱼", "麻辣味江团", "麻辣味海鲈鱼"]

base = (
    df.loc[df["菜品"] == "麻辣味烤鱼", ["日期", "销量"]]
    .groupby("日期", as_index=False)["销量"]
    .sum()
)
base["销量"] = base["销量"] / 3

new_rows = pd.concat([base.assign(菜品=flv) for flv in flavors], ignore_index=True)

df = df[df["菜品"] != "麻辣味烤鱼"]
df = pd.concat([df, new_rows], ignore_index=True)

In [24]:
# Split 'mala grilled fish' (no fish type) by date into 3 fish types: Qian fish/Jiangtuan/sea bass
flavors = ["麻辣味黔鱼", "麻辣味江团", "麻辣味海鲈鱼"]

base = (
    df.loc[df["菜品"] == "麻辣烤鱼", ["日期", "销量"]].groupby("日期", as_index=False)["销量"].sum()
)
base["销量"] = base["销量"] / 3

new_rows = pd.concat([base.assign(菜品=flv) for flv in flavors], ignore_index=True)

df = df[df["菜品"] != "麻辣烤鱼"]
df = pd.concat([df, new_rows], ignore_index=True)

In [25]:
# Split 'classic mala' (no fish type) by date into 3 fish types: Qian fish/Jiangtuan/sea bass
flavors = ["麻辣味黔鱼", "麻辣味江团", "麻辣味海鲈鱼"]

base = (
    df.loc[df["菜品"] == "经典麻辣味", ["日期", "销量"]]
    .groupby("日期", as_index=False)["销量"]
    .sum()
)
base["销量"] = base["销量"] / 3

new_rows = pd.concat([base.assign(菜品=flv) for flv in flavors], ignore_index=True)

df = df[df["菜品"] != "经典麻辣味"]
df = pd.concat([df, new_rows], ignore_index=True)

In [26]:
# Split 'garlic grilled fish' (no fish type) by date into 3 fish types: Qian fish/Jiangtuan/sea bass
flavors = ["蒜香味黔鱼", "蒜香味江团", "蒜香味海鲈鱼"]

base = (
    df.loc[df["菜品"] == "蒜香味烤鱼", ["日期", "销量"]]
    .groupby("日期", as_index=False)["销量"]
    .sum()
)
base["销量"] = base["销量"] / 3

new_rows = pd.concat([base.assign(菜品=flv) for flv in flavors], ignore_index=True)

df = df[df["菜品"] != "蒜香味烤鱼"]
df = pd.concat([df, new_rows], ignore_index=True)

In [27]:
# Split 'pickled mustard greens grilled fish' (no fish type) by date into 3 fish types: Qian fish/Jiangtuan/sea bass
flavors = ["酸菜味黔鱼", "酸菜味江团", "酸菜味海鲈鱼"]

base = (
    df.loc[df["菜品"] == "酸菜烤鱼", ["日期", "销量"]].groupby("日期", as_index=False)["销量"].sum()
)
base["销量"] = base["销量"] / 3

new_rows = pd.concat([base.assign(菜品=flv) for flv in flavors], ignore_index=True)

df = df[df["菜品"] != "酸菜烤鱼"]
df = pd.concat([df, new_rows], ignore_index=True)

In [28]:
# Split 'pickled mustard greens grilled fish' (no fish type) by date into 3 fish types: Qian fish/Jiangtuan/sea bass
flavors = ["酸菜味黔鱼", "酸菜味江团", "酸菜味海鲈鱼"]

base = (
    df.loc[df["菜品"] == "酸菜味烤鱼", ["日期", "销量"]]
    .groupby("日期", as_index=False)["销量"]
    .sum()
)
base["销量"] = base["销量"] / 3

new_rows = pd.concat([base.assign(菜品=flv) for flv in flavors], ignore_index=True)

df = df[df["菜品"] != "酸菜味烤鱼"]
df = pd.concat([df, new_rows], ignore_index=True)

In [29]:
# Split 'green pepper grilled fish' (no fish type) by date into 3 fish types: Qian fish/Jiangtuan/sea bass
flavors = ["青椒味黔鱼", "青椒味江团", "青椒味海鲈鱼"]

base = (
    df.loc[df["菜品"] == "青椒烤鱼", ["日期", "销量"]].groupby("日期", as_index=False)["销量"].sum()
)
base["销量"] = base["销量"] / 3

new_rows = pd.concat([base.assign(菜品=flv) for flv in flavors], ignore_index=True)

df = df[df["菜品"] != "青椒烤鱼"]
df = pd.concat([df, new_rows], ignore_index=True)

In [30]:
# Split 'green pepper grilled fish' (no fish type) by date into 3 fish types: Qian fish/Jiangtuan/sea bass
flavors = ["青椒味黔鱼", "青椒味江团", "青椒味海鲈鱼"]

base = (
    df.loc[df["菜品"] == "青椒味烤鱼", ["日期", "销量"]]
    .groupby("日期", as_index=False)["销量"]
    .sum()
)
base["销量"] = base["销量"] / 3

new_rows = pd.concat([base.assign(菜品=flv) for flv in flavors], ignore_index=True)

df = df[df["菜品"] != "青椒味烤鱼"]
df = pd.concat([df, new_rows], ignore_index=True)

In [31]:
df.loc[df["菜品"] == "麻辣黔鱼", "菜品"] = "麻辣味黔鱼"
df.loc[df["菜品"] == "蒜香黔鱼", "菜品"] = "蒜香味黔鱼"
df.loc[df["菜品"] == "酸菜黔鱼", "菜品"] = "酸菜味黔鱼"
df.loc[df["菜品"] == "青椒黔鱼", "菜品"] = "青椒味黔鱼"
df.loc[df["菜品"] == "经典麻辣江团", "菜品"] = "麻辣味江团"

In [32]:
# Drop rows where dish is 'lime sour-spicy', 'red lantern grilled fish', 'big red lantern Qian fish', or 'big red lantern pickled chili'
to_drop = ["青柠酸辣", "红灯笼味烤鱼", "大红灯笼味黔鱼", "大红灯笼泡椒味"]
df = df[~df["菜品"].isin(to_drop)]

In [33]:
# Same date + same dish: sum sales into one row
df = df.groupby(["日期", "菜品"], as_index=False)["销量"].sum()
df = df.sort_values(["日期", "菜品"], na_position="last")
df.head()

,日期,菜品,销量
0,2025-05-01,掌中宝,66.000000
1,2025-05-01,烤羊肉串,168.000000
2,2025-05-01,老坛酸菜味江团,2.666667
3,2025-05-01,老坛酸菜味海鲈鱼,2.666667
4,2025-05-01,老坛酸菜味黔鱼,2.666667


In [34]:
df["菜品"].unique()

array(['掌中宝', '烤羊肉串', '老坛酸菜味江团', '老坛酸菜味海鲈鱼', '老坛酸菜味黔鱼', '蒜香味江团', '蒜香味海鲈鱼',
       '蒜香味黔鱼', '酸菜味江团', '酸菜味海鲈鱼', '酸菜味黔鱼', '青椒味江团', '青椒味海鲈鱼', '青椒味黔鱼',
       '麻辣味江团', '麻辣味海鲈鱼', '麻辣味黔鱼'], dtype=object)

In [35]:
# Overwrite the original file (this will modify/replace the dataset)
path = "food_sales.csv"
df.to_csv(path, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")